In [3]:
# =============================================================================
# MINDSHIFT ANALYTICS HAULMARK CHALLENGE — v4 PIPELINE
#
# Key upgrade vs v3: CATEGORICAL FEATURE ENGINEERING
#   operator_id  → driving-style aggregations per shift
#   vehicle      → handoff count, vehicle×mine combo encoding
#   mine_anon    → terrain aggregations, operator×mine combo encoding
#
# ── New features added (10) ──────────────────────────────────────────────────
#   OPERATOR STYLE:
#     operator_idle_ratio    – idle fraction for this operator this shift
#     operator_accel_score   – accel events per travel hour (aggression)
#     operator_avg_speed     – mean speed while moving (km/h style)
#     operator_dump_rate     – dumps per travel hour (efficiency)
#     operator_te            – OOF target encoding of operator_id
#
#   MINE TERRAIN:
#     mine_grade_severity    – mean uphill grade ratio at this mine this shift
#     mine_distance_per_hr   – haul distance per engine-on hour (route intensity)
#
#   COMBO / INTERACTION:
#     n_operators_in_shift   – distinct operators on this truck this shift
#     vehicle_mine_te        – TE of vehicle+mine combo (assignment pattern)
#     operator_mine_te       – TE of operator+mine combo (style × terrain)
#
# ── Total features: 25 ──────────────────────────────────────────────────────
# =============================================================================

import pandas as pd
import numpy as np
import lightgbm as lgb
import xgboost as xgb
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.isotonic import IsotonicRegression
from sklearn.model_selection import KFold
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
from scipy.optimize import minimize
from pathlib import Path
from sklearn.metrics import mean_squared_error

# =============================================================================
# CONFIG
# =============================================================================

BASE       = Path("/kaggle/input/competitions/mindshift-analytics-haul-mark-challenge")
OUTPUT_DIR = Path("/kaggle/working")
OUTPUT_DIR.mkdir(exist_ok=True)

TRAIN_FILES = [
    BASE / "telemetry_2026-01-01_2026-01-10.parquet",
    BASE / "telemetry_2026-01-11_2026-01-20.parquet",
    BASE / "telemetry_2026-02-01_2026-02-10.parquet",
    BASE / "telemetry_2026-02-11_2026-02-20.parquet",
    BASE / "telemetry_2026-03-01_2026-03-11.parquet",
]
TEST_FILES = [
    BASE / "telemetry_2026-01-21_2026-01-31.parquet",
    BASE / "telemetry_2026-02-21_2026-02-28.parquet",
    BASE / "telemetry_2026-03-12_2026-03-20.parquet",
]

TARGETS_FILE = Path("/kaggle/input/datasets/nandishrajukori/train-targets-csv/train_targets.csv")
ID_MAPPING   = BASE / "id_mapping_new.csv"

MIN_SATELLITES = 4
MAX_HDOP       = 1.5
MAX_PDOP       = 2.0
MAX_GAP_SEC    = 300
DUMP_VOLTAGE   = 2.5
GRADE_PENALTY  = 4.0

SHIFT_ORDER = ["C", "A", "B"]
N_SHIFTS    = 3
EWMA_ALPHA  = 0.3
EWMA_BLEND  = 0.15

CV_FOLDS = [
    (
        ["2026-01-01","2026-01-02","2026-01-03","2026-01-04","2026-01-05",
         "2026-01-06","2026-01-07","2026-01-08","2026-01-09","2026-01-10"],
        ["2026-01-11","2026-01-12","2026-01-13","2026-01-14","2026-01-15"],
    ),
    (
        ["2026-01-01","2026-01-02","2026-01-03","2026-01-04","2026-01-05",
         "2026-01-06","2026-01-07","2026-01-08","2026-01-09","2026-01-10",
         "2026-01-11","2026-01-12","2026-01-13","2026-01-14","2026-01-15"],
        ["2026-01-16","2026-01-17","2026-01-18","2026-01-19","2026-01-20"],
    ),
    (
        ["2026-02-01","2026-02-02","2026-02-03","2026-02-04","2026-02-05",
         "2026-02-06","2026-02-07","2026-02-08","2026-02-09","2026-02-10"],
        ["2026-02-11","2026-02-12","2026-02-13","2026-02-14","2026-02-15"],
    ),
]

OPTUNA_TRIALS   = 35
N_ENSEMBLE_RUNS = 5
SMOOTHING_K     = 10

# =============================================================================
# STEP 1 — DATA LOADING
#   Now also loads operator_id in addition to existing columns
# =============================================================================

def assign_op_day(df, ts_col="ts"):
    ts = df[ts_col]
    ts = ts.dt.tz_convert("Asia/Kolkata") if ts.dt.tz else ts.dt.tz_localize("Asia/Kolkata")
    df["op_day"]  = (ts + pd.Timedelta(hours=2)).dt.date.astype(str)
    df["_ts_ist"] = ts
    return df


def assign_shift(df):
    h = df["_ts_ist"].dt.hour
    df["shift"] = np.select(
        [(h >= 22) | (h < 6),
         (h >= 6)  & (h < 14),
         (h >= 14) & (h < 22)],
        SHIFT_ORDER, default="A",
    )
    return df


def load_telemetry(file_list, gps_filter=True):
    NEEDED = [
        "ts", "vehicle", "mine_anon", "operator_id",   # ← operator_id added
        "speed", "ignition", "altitude",
        "disthav", "analog_input_1",
        "satellites", "gnss_hdop", "gnss_pdop",
    ]
    chunks = []
    for f in file_list:
        print(f"  Loading {Path(f).name} ...", end=" ")
        all_cols = pd.read_parquet(f, engine="pyarrow").columns.tolist()
        use_cols = [c for c in NEEDED if c in all_cols]
        chunk    = pd.read_parquet(f, engine="pyarrow", columns=use_cols)
        chunk["ts"] = pd.to_datetime(chunk["ts"], utc=True)
        # ensure operator_id exists even if column is absent in some files
        if "operator_id" not in chunk.columns:
            chunk["operator_id"] = "unknown"
        chunks.append(chunk)
        print(f"{len(chunk):,} rows")

    df = pd.concat(chunks, ignore_index=True)
    df = df[df["vehicle"].str.startswith("Dump")].copy()
    df["operator_id"] = df["operator_id"].fillna("unknown").astype(str)

    if gps_filter:
        before = len(df)
        if "satellites" in df.columns:
            df = df[df["satellites"].isna() | (df["satellites"] >= MIN_SATELLITES)]
        if "gnss_hdop" in df.columns:
            df = df[df["gnss_hdop"].isna()  | (df["gnss_hdop"]  <= MAX_HDOP)]
        if "gnss_pdop" in df.columns:
            df = df[df["gnss_pdop"].isna()  | (df["gnss_pdop"]  <= MAX_PDOP)]
        print(f"  GPS filter removed {before - len(df):,} rows "
              f"({(before - len(df)) / before * 100:.1f}%)")

    df = assign_op_day(df)
    df = assign_shift(df)
    df = df.sort_values(["vehicle", "op_day", "shift", "ts"]).reset_index(drop=True)
    print(f"  → {len(df):,} rows | {df['vehicle'].nunique()} dumpers | "
          f"{df['operator_id'].nunique()} operators")
    return df

# =============================================================================
# STEP 2 — ACTIVITY-WEIGHTED TARGET SPLIT
# =============================================================================

def build_shift_targets(targets_daily, shift_feats):
    print("  Building activity-weighted shift targets ...")
    day_dist = (
        shift_feats.groupby(["vehicle", "op_day"])["equiv_flat_distance"]
        .transform("sum").rename("day_dist")
    )
    sf = shift_feats.copy()
    sf["day_dist"]   = day_dist
    sf["dist_share"] = sf["equiv_flat_distance"] / (sf["day_dist"] + 1e-9)

    no_act = sf["day_dist"] < 1.0
    sf.loc[no_act, "dist_share"] = 1.0 / N_SHIFTS

    share_sum = sf.groupby(["vehicle", "op_day"])["dist_share"].transform("sum")
    sf["dist_share"] = sf["dist_share"] / (share_sum + 1e-9)

    merged = sf[["vehicle", "op_day", "shift", "dist_share"]].merge(
        targets_daily, on=["vehicle", "op_day"], how="left"
    )
    merged["fuel_consumption_L"] = merged["fuel_consumption_L"] * merged["dist_share"]
    print(f"  → {len(merged):,} shift-level target rows")
    return merged[["vehicle", "op_day", "shift", "fuel_consumption_L"]]

# =============================================================================
# STEP 3 — FEATURE ENGINEERING
#   Adds 10 new categorical aggregation features on top of original 15
# =============================================================================

def build_features(df):
    print("  Engineering shift-level features ...")
    GROUP = ["vehicle", "op_day", "shift"]
    eps   = 1e-9

    # ── time deltas ──────────────────────────────────────────────────────────
    df["dt_sec"] = (
        df.groupby(GROUP)["ts"]
        .transform(lambda x: x.diff().dt.total_seconds().shift(-1))
        .clip(0, MAX_GAP_SEC).fillna(0)
    )

    # ── altitude / distance ──────────────────────────────────────────────────
    df["dz"]        = df.groupby(GROUP)["altitude"].transform(lambda x: x.diff().fillna(0))
    df["is_moving"] = (df["speed"] >= 2.0).astype(float)
    df["dz_up"]     = df["dz"].clip(lower=0) * df["is_moving"]
    df["dx"]        = df["disthav"].fillna(0)
    df["dx_up"]     = df["dx"] * (df["dz"] > 0).astype(float)

    # ── acceleration proxy ───────────────────────────────────────────────────
    df["dv"]    = df.groupby(GROUP)["speed"].transform(lambda x: x.diff().fillna(0))
    df["dv_up"] = df["dv"].clip(lower=0)

    # ── engine state ─────────────────────────────────────────────────────────
    df["idle_sec"]   = df["dt_sec"] * ((df["ignition"] == 1) & (df["speed"] < 1.0)).astype(float)
    df["travel_sec"] = df["dt_sec"] * ((df["ignition"] == 1) & (df["speed"] >= 1.0)).astype(float)

    # ── dump events ──────────────────────────────────────────────────────────
    if "analog_input_1" in df.columns:
        df["above_dump"] = (df["analog_input_1"] > DUMP_VOLTAGE).astype(float)
        df["dump_evt"]   = (
            (df["above_dump"] == 1) &
            (df.groupby("vehicle")["above_dump"].shift(1).fillna(0) == 0)
        ).astype(int)
    else:
        df["dump_evt"] = 0

    # ── SHIFT-LEVEL BASE AGGREGATION ─────────────────────────────────────────
    agg = df.groupby(GROUP).agg(
        total_idle_hr      = ("idle_sec",   lambda x: x.sum() / 3600),
        total_travel_hr    = ("travel_sec", lambda x: x.sum() / 3600),
        total_time_hr      = ("dt_sec",     lambda x: x.sum() / 3600),
        total_distance     = ("dx",         "sum"),
        total_uphill_z     = ("dz_up",      "sum"),
        total_uphill_dx    = ("dx_up",      "sum"),
        dump_count         = ("dump_evt",   "sum"),
        total_accel_events = ("dv_up",      "sum"),
    ).reset_index()

    # ── dominant mine per shift ───────────────────────────────────────────────
    if "mine_anon" in df.columns:
        mine_agg = (
            df.groupby(GROUP)["mine_anon"]
            .agg(lambda x: x.mode().iloc[0] if len(x) > 0 else "unknown")
            .reset_index().rename(columns={"mine_anon": "mine_str"})
        )
        agg = agg.merge(mine_agg, on=GROUP, how="left")
    else:
        agg["mine_str"] = "unknown"

    # ── dominant operator per shift ───────────────────────────────────────────
    if "operator_id" in df.columns:
        op_dom = (
            df.groupby(GROUP)["operator_id"]
            .agg(lambda x: x.mode().iloc[0] if len(x) > 0 else "unknown")
            .reset_index().rename(columns={"operator_id": "operator_str"})
        )
        agg = agg.merge(op_dom, on=GROUP, how="left")
    else:
        agg["operator_str"] = "unknown"

    # ── physics features (original) ───────────────────────────────────────────
    agg["total_idle_hr"]      = agg["total_idle_hr"].clip(upper=8.5)
    agg["total_travel_hr"]    = agg["total_travel_hr"].clip(upper=8.5)
    agg["uphill_grade_ratio"] = agg["total_uphill_z"] / (agg["total_uphill_dx"] + eps)
    agg["equiv_flat_distance"]= agg["total_distance"] * (1.0 + agg["uphill_grade_ratio"] * GRADE_PENALTY)
    agg["loaded_climb_work"]  = agg["dump_count"] * agg["total_uphill_z"]
    agg["idle_time_ratio"]    = (agg["total_idle_hr"] / (agg["total_time_hr"] + eps)).clip(0, 1)
    agg["dump_count_x_idle"]  = agg["dump_count"] * agg["idle_time_ratio"]
    agg["uphill_z_per_dump"]  = agg["total_uphill_z"] / (agg["dump_count"] + 1)

    # =========================================================================
    # NEW — OPERATOR-LEVEL AGGREGATION FEATURES
    #   Computed per (operator_id, vehicle, op_day, shift) then merged back.
    #   This captures driving style regardless of which truck they drove.
    # =========================================================================
    if "operator_id" in df.columns:
        op_agg = df.groupby(GROUP + ["operator_id"]).agg(
            _op_idle_sec    = ("idle_sec",   "sum"),
            _op_travel_sec  = ("travel_sec", "sum"),
            _op_dv_up       = ("dv_up",      "sum"),
            _op_speed_mean  = ("speed",      lambda x: x[x >= 2.0].mean() if (x >= 2.0).any() else 0.0),
            _op_dump        = ("dump_evt",   "sum"),
        ).reset_index()

        # dominant operator aggregated stats per shift
        op_shift = (
            op_agg.sort_values("_op_travel_sec", ascending=False)
            .groupby(GROUP).first()          # keep dominant operator's stats
            .reset_index()
        )
        op_shift["operator_idle_ratio"] = (
            op_shift["_op_idle_sec"] /
            (op_shift["_op_idle_sec"] + op_shift["_op_travel_sec"] + eps)
        ).clip(0, 1)
        op_shift["operator_accel_score"] = (
            op_shift["_op_dv_up"] /
            (op_shift["_op_travel_sec"] / 3600 + eps)
        ).clip(0)
        op_shift["operator_avg_speed"] = op_shift["_op_speed_mean"].clip(0)
        op_shift["operator_dump_rate"] = (
            op_shift["_op_dump"] /
            (op_shift["_op_travel_sec"] / 3600 + eps)
        ).clip(0)

        agg = agg.merge(
            op_shift[GROUP + ["operator_idle_ratio", "operator_accel_score",
                               "operator_avg_speed",  "operator_dump_rate"]],
            on=GROUP, how="left",
        )

        # ── number of distinct operators on this truck this shift ─────────────
        n_ops = (
            df[df["operator_id"] != "unknown"]
            .groupby(GROUP)["operator_id"]
            .nunique()
            .reset_index()
            .rename(columns={"operator_id": "n_operators_in_shift"})
        )
        agg = agg.merge(n_ops, on=GROUP, how="left")
        agg["n_operators_in_shift"] = agg["n_operators_in_shift"].fillna(1).clip(1, 5)
    else:
        agg["operator_idle_ratio"]   = 0.0
        agg["operator_accel_score"]  = 0.0
        agg["operator_avg_speed"]    = 0.0
        agg["operator_dump_rate"]    = 0.0
        agg["n_operators_in_shift"]  = 1.0

    # =========================================================================
    # NEW — MINE-LEVEL AGGREGATION FEATURES
    #   Terrain character of the mine this shift.
    # =========================================================================
    if "mine_anon" in df.columns:
        mine_agg2 = df.groupby(GROUP).agg(
            _mine_grade_num = ("dz_up", "sum"),
            _mine_grade_den = ("dx_up", "sum"),
            _mine_dist      = ("dx",    "sum"),
            _mine_eng_sec   = ("dt_sec", lambda x: x[(df.loc[x.index, "ignition"] == 1)].sum()
                                if "ignition" in df.columns else x.sum()),
        ).reset_index()

        mine_agg2["mine_grade_severity"] = (
            mine_agg2["_mine_grade_num"] /
            (mine_agg2["_mine_grade_den"] + eps)
        ).clip(0, 1)
        mine_agg2["mine_distance_per_hr"] = (
            mine_agg2["_mine_dist"] /
            (mine_agg2["_mine_eng_sec"] / 3600 + eps)
        ).clip(0)

        agg = agg.merge(
            mine_agg2[GROUP + ["mine_grade_severity", "mine_distance_per_hr"]],
            on=GROUP, how="left",
        )
    else:
        agg["mine_grade_severity"]  = 0.0
        agg["mine_distance_per_hr"] = 0.0

    # =========================================================================
    # NEW — COMBO KEYS  (used for target encoding later)
    # =========================================================================
    agg["vehicle_mine_key"]    = agg["vehicle"].astype(str) + "__" + agg["mine_str"].astype(str)
    agg["operator_mine_key"]   = agg["operator_str"].astype(str) + "__" + agg["mine_str"].astype(str)

    agg = agg.replace([np.inf, -np.inf], np.nan).fillna(0)
    print(f"  → {agg.shape[0]:,} shift-rows | {agg.shape[1]} columns")
    return agg

# =============================================================================
# STEP 4 — LAG FEATURES
# =============================================================================

SHIFT_RANK = {"C": 0, "A": 1, "B": 2}
LAG_COLS   = ["loaded_climb_work", "dump_count", "idle_time_ratio", "equiv_flat_distance"]

def add_lag_features(df):
    print("  Adding lag features ...")
    df = df.copy()
    df["shift_rank"] = df["shift"].map(SHIFT_RANK)
    df = df.sort_values(["vehicle", "op_day", "shift_rank"]).reset_index(drop=True)
    for col in LAG_COLS:
        df[f"lag1_{col}"] = df.groupby("vehicle")[col].shift(1).fillna(0)
        df[f"lag2_{col}"] = df.groupby("vehicle")[col].shift(2).fillna(0)
    df.drop(columns=["shift_rank"], inplace=True)
    return df

# =============================================================================
# STEP 5 — EWMA ROLLING FUEL PRIOR
# =============================================================================

def add_ewma_prior(feat_df, targets_daily):
    print("  Computing EWMA rolling fuel prior ...")
    daily = targets_daily[["vehicle", "op_day", "fuel_consumption_L"]].copy()
    daily["fuel_per_shift"] = daily["fuel_consumption_L"] / N_SHIFTS
    daily = daily.sort_values(["vehicle", "op_day"])

    def vehicle_ewma(grp):
        grp = grp.sort_values("op_day")
        grp["ewma_prior"] = (
            grp["fuel_per_shift"].shift(1)
            .ewm(alpha=EWMA_ALPHA, adjust=True).mean()
            .fillna(grp["fuel_per_shift"].mean())
        )
        return grp[["vehicle", "op_day", "ewma_prior"]]

    ewma_df = (
        daily.groupby("vehicle", group_keys=False)
        .apply(vehicle_ewma).reset_index(drop=True)
    )
    feat_df = feat_df.merge(ewma_df, on=["vehicle", "op_day"], how="left")
    global_mean = daily["fuel_per_shift"].mean()
    feat_df["ewma_fuel_prior"] = feat_df["ewma_prior"].fillna(global_mean)
    feat_df.drop(columns=["ewma_prior"], inplace=True, errors="ignore")
    return feat_df

# =============================================================================
# STEP 6 — CV-SAFE TARGET ENCODING
#   Now encodes: vehicle, mine_str, operator_str, vehicle_mine_key,
#                operator_mine_key  (all OOF-safe inside CV folds)
# =============================================================================

# All categorical columns that get target-encoded
TE_COLS = ["vehicle", "mine_str", "operator_str", "vehicle_mine_key", "operator_mine_key"]


def _smooth_encode(train_df, col, target_col, global_mean):
    """Smoothed target encoding map from a training slice."""
    stats = train_df.groupby(col)[target_col].agg(["sum", "count"]).reset_index()
    stats["encoded"] = (
        (stats["sum"] + SMOOTHING_K * global_mean) / (stats["count"] + SMOOTHING_K)
    )
    return stats.set_index(col)["encoded"].to_dict()


def apply_target_encodings_train(train_df, val_df):
    """Encode val using full train stats (fold-level, no leakage into val)."""
    global_mean = train_df["fuel_consumption_L"].mean()
    val_df = val_df.copy()
    for col in TE_COLS:
        enc_map = _smooth_encode(train_df, col, "fuel_consumption_L", global_mean)
        val_df[f"{col}_te"] = val_df[col].map(enc_map).fillna(global_mean)
    return val_df


def build_full_target_encodings(full_train_df):
    """Build encoding maps on entire training set for final model."""
    global_mean = full_train_df["fuel_consumption_L"].mean()
    enc_maps = {}
    for col in TE_COLS:
        enc_maps[col] = _smooth_encode(full_train_df, col, "fuel_consumption_L", global_mean)
    return enc_maps, global_mean


def apply_full_encodings(df, enc_maps, global_mean):
    df = df.copy()
    for col in TE_COLS:
        df[f"{col}_te"] = df[col].map(enc_maps[col]).fillna(global_mean)
    return df


def oof_target_encode_train(tr_raw):
    """
    OOF K-Fold target encoding for the training split itself.
    Prevents leakage when training the model on train rows.
    """
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    global_mean = tr_raw["fuel_consumption_L"].mean()

    for col in TE_COLS:
        tr_raw[f"{col}_te"] = np.nan

    for tr_idx, oof_idx in kf.split(tr_raw):
        fold_tr = tr_raw.iloc[tr_idx]
        gm      = fold_tr["fuel_consumption_L"].mean()
        for col in TE_COLS:
            enc = _smooth_encode(fold_tr, col, "fuel_consumption_L", gm)
            col_loc = tr_raw.columns.get_loc(f"{col}_te")
            tr_raw.iloc[oof_idx, col_loc] = tr_raw.iloc[oof_idx][col].map(enc)

    for col in TE_COLS:
        tr_raw[f"{col}_te"] = tr_raw[f"{col}_te"].fillna(global_mean)

    return tr_raw

# =============================================================================
# MODEL FEATURES — 25 total
#   Original 15  +  10 new categorical aggregation features
# =============================================================================

MODEL_FEATURES = [
    # ── Original physics (Tier 1-3) ─────────────────────────────────────────
    "total_idle_hr", "total_travel_hr", "equiv_flat_distance",
    "loaded_climb_work", "dump_count", "total_accel_events",
    "dump_count_x_idle", "uphill_z_per_dump", "idle_time_ratio",
    "lag1_loaded_climb_work", "lag1_dump_count", "lag1_idle_time_ratio",
    # ── Original target encodings (Tier 4) ───────────────────────────────────
    "vehicle_te", "mine_str_te", "ewma_fuel_prior",
    # ── NEW: operator driving style ──────────────────────────────────────────
    "operator_idle_ratio", "operator_accel_score",
    "operator_avg_speed",  "operator_dump_rate",
    "n_operators_in_shift",
    # ── NEW: mine terrain ────────────────────────────────────────────────────
    "mine_grade_severity", "mine_distance_per_hr",
    # ── NEW: combo target encodings ──────────────────────────────────────────
    "operator_str_te", "vehicle_mine_key_te", "operator_mine_key_te",
]

def get_feature_cols(df):
    return [f for f in MODEL_FEATURES if f in df.columns]

# =============================================================================
# LOG-TRANSFORM HELPERS
# =============================================================================

def log_transform(y): return np.log1p(y)
def inv_transform(y): return np.expm1(y).clip(0)

# =============================================================================
# STEP 7 — OPTUNA OBJECTIVES
# =============================================================================

def lgb_objective(trial, X_tr, y_tr_log, X_val, y_val_raw, quantile=False):
    obj = "quantile" if quantile else "regression"
    met = "quantile" if quantile else "rmse"
    params = {
        "objective":         obj, "metric": met, "alpha": 0.5,
        "verbosity":         -1,  "seed":   42,
        "learning_rate":     trial.suggest_float("lr",        0.01, 0.05, log=True),
        "num_leaves":        trial.suggest_int("num_leaves",  8,    32),
        "min_child_samples": trial.suggest_int("min_child",   10,   50),
        "feature_fraction":  trial.suggest_float("feat_frac", 0.6,  1.0),
        "bagging_fraction":  trial.suggest_float("bag_frac",  0.6,  1.0),
        "bagging_freq":      5,
        "lambda_l1":         trial.suggest_float("l1",        0.1,  10.0, log=True),
        "lambda_l2":         trial.suggest_float("l2",        0.1,  10.0, log=True),
    }
    dtrain = lgb.Dataset(X_tr,  label=y_tr_log,            free_raw_data=False)
    dval   = lgb.Dataset(X_val, label=log_transform(y_val_raw), reference=dtrain, free_raw_data=False)
    model  = lgb.train(
        params, dtrain, num_boost_round=500, valid_sets=[dval],
        callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(0)],
    )
    return np.sqrt(mean_squared_error(y_val_raw, inv_transform(model.predict(X_val))))


def xgb_objective(trial, X_tr, y_tr_log, X_val, y_val_raw):
    params = {
        "objective": "reg:squarederror", "verbosity": 0, "seed": 42,
        "learning_rate":    trial.suggest_float("lr",        0.01, 0.05, log=True),
        "max_depth":        trial.suggest_int("max_depth",   2,    6),
        "min_child_weight": trial.suggest_int("min_child",   10,   40),
        "subsample":        trial.suggest_float("subsample", 0.6,  1.0),
        "colsample_bytree": trial.suggest_float("colsample", 0.7,  1.0),
        "reg_alpha":        trial.suggest_float("alpha",     0.1,  10.0, log=True),
        "reg_lambda":       trial.suggest_float("lambda_",   0.1,  10.0, log=True),
        "n_estimators": 500, "early_stopping_rounds": 50,
        "tree_method": "hist", "device": "cpu",
    }
    model = xgb.XGBRegressor(**params)
    model.fit(X_tr, y_tr_log, eval_set=[(X_val, log_transform(y_val_raw))], verbose=False)
    return np.sqrt(mean_squared_error(y_val_raw, inv_transform(model.predict(X_val))))


def ridge_objective(trial, X_tr, y_tr_log, X_val, y_val_raw):
    alpha = trial.suggest_float("alpha", 0.1, 500.0, log=True)
    pipe  = Pipeline([("scaler", StandardScaler()), ("ridge", Ridge(alpha=alpha))])
    pipe.fit(X_tr, y_tr_log)
    return np.sqrt(mean_squared_error(y_val_raw, inv_transform(pipe.predict(X_val))))

# =============================================================================
# STEP 8 — WALK-FORWARD CV
# =============================================================================

def walk_forward_cv(train_df, targets_daily, feature_cols_base):
    print("\n=== Walk-forward CV + Optuna tuning ===")

    fold_results = []
    best_params  = {"lgb": None, "lgb_q": None, "xgb": None, "ridge": None}
    oof_records  = []

    for i, (train_days, val_days) in enumerate(CV_FOLDS):
        tr_raw  = train_df[train_df["op_day"].isin(train_days)].copy()
        val_raw = train_df[train_df["op_day"].isin(val_days)].copy()
        tr_raw  = tr_raw [tr_raw ["fuel_consumption_L"].notna()]
        val_raw = val_raw[val_raw["fuel_consumption_L"].notna()]

        if len(tr_raw) == 0 or len(val_raw) == 0:
            print(f"  Fold {i+1}: skipped"); continue

        tr_daily_fold = targets_daily[targets_daily["op_day"].isin(train_days)]
        tr_raw  = add_ewma_prior(tr_raw,  tr_daily_fold)
        val_raw = add_ewma_prior(val_raw, tr_daily_fold)

        # val encoded using full fold-train stats (no leakage into val)
        val_raw = apply_target_encodings_train(tr_raw, val_raw)

        # train encoded OOF (no self-leakage)
        tr_raw = oof_target_encode_train(tr_raw)

        feature_cols = get_feature_cols(tr_raw)
        for col in feature_cols:
            if col not in val_raw.columns:
                val_raw[col] = 0

        X_tr,  y_tr_raw  = tr_raw [feature_cols], tr_raw ["fuel_consumption_L"].values
        X_val, y_val_raw = val_raw[feature_cols], val_raw["fuel_consumption_L"].values
        y_tr_log = log_transform(y_tr_raw)

        print(f"\n  ── Fold {i+1}  (train={len(tr_raw)}, val={len(val_raw)}, "
              f"features={len(feature_cols)}) ──")

        # ── LGB RMSE ──────────────────────────────────────────────────────────
        print("    Tuning LGB RMSE ...")
        study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=42))
        study.optimize(lambda t: lgb_objective(t, X_tr, y_tr_log, X_val, y_val_raw, False),
                       n_trials=OPTUNA_TRIALS, show_progress_bar=False)
        lgb_best = study.best_params
        if best_params["lgb"] is None: best_params["lgb"] = lgb_best

        lgb_p = {
            "objective": "regression", "metric": "rmse", "verbosity": -1, "seed": 42,
            "learning_rate": lgb_best["lr"], "num_leaves": lgb_best["num_leaves"],
            "min_child_samples": lgb_best["min_child"], "feature_fraction": lgb_best["feat_frac"],
            "bagging_fraction": lgb_best["bag_frac"], "bagging_freq": 5,
            "lambda_l1": lgb_best["l1"], "lambda_l2": lgb_best["l2"],
        }
        dtrain  = lgb.Dataset(X_tr,  label=y_tr_log, free_raw_data=False)
        dval_ds = lgb.Dataset(X_val, label=log_transform(y_val_raw), reference=dtrain, free_raw_data=False)
        lgb_m   = lgb.train(lgb_p, dtrain, num_boost_round=1000, valid_sets=[dval_ds],
                            callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(0)])
        lgb_preds = inv_transform(lgb_m.predict(X_val))
        lgb_rmse  = np.sqrt(mean_squared_error(y_val_raw, lgb_preds))
        print(f"    LGB RMSE  = {lgb_rmse:.2f} L")

        # ── LGB Quantile ──────────────────────────────────────────────────────
        print("    Tuning LGB Quantile ...")
        study_q = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=99))
        study_q.optimize(lambda t: lgb_objective(t, X_tr, y_tr_log, X_val, y_val_raw, True),
                         n_trials=OPTUNA_TRIALS, show_progress_bar=False)
        lgb_q_best = study_q.best_params
        if best_params["lgb_q"] is None: best_params["lgb_q"] = lgb_q_best

        lgb_qp = {
            "objective": "quantile", "metric": "quantile", "alpha": 0.5,
            "verbosity": -1, "seed": 42,
            "learning_rate": lgb_q_best["lr"], "num_leaves": lgb_q_best["num_leaves"],
            "min_child_samples": lgb_q_best["min_child"], "feature_fraction": lgb_q_best["feat_frac"],
            "bagging_fraction": lgb_q_best["bag_frac"], "bagging_freq": 5,
            "lambda_l1": lgb_q_best["l1"], "lambda_l2": lgb_q_best["l2"],
        }
        lgb_q_m    = lgb.train(lgb_qp, dtrain, num_boost_round=1000, valid_sets=[dval_ds],
                               callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(0)])
        lgb_q_preds = inv_transform(lgb_q_m.predict(X_val))
        lgb_q_rmse  = np.sqrt(mean_squared_error(y_val_raw, lgb_q_preds))
        print(f"    LGB Q50   = {lgb_q_rmse:.2f} L")

        # ── XGBoost ───────────────────────────────────────────────────────────
        print("    Tuning XGBoost ...")
        study_x = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=42))
        study_x.optimize(lambda t: xgb_objective(t, X_tr, y_tr_log, X_val, y_val_raw),
                         n_trials=OPTUNA_TRIALS, show_progress_bar=False)
        xgb_best = study_x.best_params
        if best_params["xgb"] is None: best_params["xgb"] = xgb_best

        xgb_m = xgb.XGBRegressor(
            objective="reg:squarederror", verbosity=0, seed=42,
            learning_rate=xgb_best["lr"], max_depth=xgb_best["max_depth"],
            min_child_weight=xgb_best["min_child"], subsample=xgb_best["subsample"],
            colsample_bytree=xgb_best["colsample"], reg_alpha=xgb_best["alpha"],
            reg_lambda=xgb_best["lambda_"], n_estimators=1000,
            early_stopping_rounds=50, tree_method="hist",
        )
        xgb_m.fit(X_tr, y_tr_log, eval_set=[(X_val, log_transform(y_val_raw))], verbose=False)
        xgb_preds = inv_transform(xgb_m.predict(X_val))
        xgb_rmse  = np.sqrt(mean_squared_error(y_val_raw, xgb_preds))
        print(f"    XGB RMSE  = {xgb_rmse:.2f} L")

        # ── Ridge ─────────────────────────────────────────────────────────────
        print("    Tuning Ridge ...")
        study_r = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=42))
        study_r.optimize(lambda t: ridge_objective(t, X_tr, y_tr_log, X_val, y_val_raw),
                         n_trials=OPTUNA_TRIALS, show_progress_bar=False)
        ridge_best = study_r.best_params
        if best_params["ridge"] is None: best_params["ridge"] = ridge_best

        ridge_pipe = Pipeline([("scaler", StandardScaler()), ("ridge", Ridge(alpha=ridge_best["alpha"]))])
        ridge_pipe.fit(X_tr, y_tr_log)
        ridge_preds = inv_transform(ridge_pipe.predict(X_val))
        ridge_rmse  = np.sqrt(mean_squared_error(y_val_raw, ridge_preds))
        print(f"    Ridge RMSE = {ridge_rmse:.2f} L")

        for j in range(len(y_val_raw)):
            oof_records.append({
                "true": y_val_raw[j], "lgb": lgb_preds[j], "lgb_q": lgb_q_preds[j],
                "xgb": xgb_preds[j], "ridge": ridge_preds[j],
            })
        fold_results.append({
            "fold": i + 1, "n_train": len(tr_raw), "n_val": len(val_raw),
            "lgb_rmse": lgb_rmse, "lgb_q_rmse": lgb_q_rmse,
            "xgb_rmse": xgb_rmse, "ridge_rmse": ridge_rmse,
        })

    oof_df = pd.DataFrame(oof_records)
    print("\n  Optimising ensemble blend weights ...")

    def blend_rmse(w):
        w = np.clip(w, 0, None); w = w / (w.sum() + 1e-9)
        blend = w[0]*oof_df["lgb"] + w[1]*oof_df["lgb_q"] + w[2]*oof_df["xgb"] + w[3]*oof_df["ridge"]
        return float(np.sqrt(mean_squared_error(oof_df["true"], blend)))

    res = minimize(blend_rmse, x0=[0.4, 0.15, 0.3, 0.15], method="Nelder-Mead",
                   options={"xatol": 1e-6, "fatol": 1e-6, "maxiter": 8000})
    raw_w            = np.clip(res.x, 0, None)
    ensemble_weights = raw_w / raw_w.sum()
    ensemble_rmse    = blend_rmse(ensemble_weights)

    print(f"  Weights → LGB={ensemble_weights[0]:.3f}  LGB-Q={ensemble_weights[1]:.3f}"
          f"  XGB={ensemble_weights[2]:.3f}  Ridge={ensemble_weights[3]:.3f}")
    print(f"  Ensemble OOF RMSE → {ensemble_rmse:.2f} L")
    return ensemble_rmse, fold_results, best_params, ensemble_weights, oof_df

# =============================================================================
# STEP 9 — TRAIN FINAL ENSEMBLE
# =============================================================================

def train_final_ensemble(train_df, feature_cols, best_params, ensemble_weights):
    print("\n=== Training final ensemble on all training data ===")

    clean = train_df[train_df["fuel_consumption_L"].notna()]
    X     = clean[feature_cols]
    y_log = log_transform(clean["fuel_consumption_L"].values)

    lgb_models, lgb_q_models, xgb_models, ridge_models = [], [], [], []

    for run in range(N_ENSEMBLE_RUNS):
        seed = 42 + run * 7
        print(f"  Run {run + 1}/{N_ENSEMBLE_RUNS}  (seed={seed})")

        lp = best_params["lgb"] or {}
        lgb_params = {
            "objective": "regression", "metric": "rmse", "verbosity": -1, "seed": seed,
            "learning_rate": lp.get("lr", 0.03), "num_leaves": lp.get("num_leaves", 16),
            "min_child_samples": lp.get("min_child", 20), "feature_fraction": lp.get("feat_frac", 0.8),
            "bagging_fraction": lp.get("bag_frac", 0.8), "bagging_freq": 5,
            "lambda_l1": lp.get("l1", 1.0), "lambda_l2": lp.get("l2", 1.0),
        }
        lgb_models.append(lgb.train(
            lgb_params, lgb.Dataset(X, label=y_log, free_raw_data=False), num_boost_round=800
        ))

        lqp = best_params["lgb_q"] or lp
        lgb_q_params = {
            **lgb_params,
            "objective": "quantile", "metric": "quantile", "alpha": 0.5,
            "learning_rate": lqp.get("lr", 0.03), "num_leaves": lqp.get("num_leaves", 16),
        }
        lgb_q_models.append(lgb.train(
            lgb_q_params, lgb.Dataset(X, label=y_log, free_raw_data=False), num_boost_round=800
        ))

        xp    = best_params["xgb"] or {}
        xgb_m = xgb.XGBRegressor(
            objective="reg:squarederror", verbosity=0, seed=seed,
            learning_rate=xp.get("lr", 0.03), max_depth=xp.get("max_depth", 4),
            min_child_weight=xp.get("min_child", 15), subsample=xp.get("subsample", 0.8),
            colsample_bytree=xp.get("colsample", 0.8), reg_alpha=xp.get("alpha", 1.0),
            reg_lambda=xp.get("lambda_", 1.0), n_estimators=800, tree_method="hist",
        )
        xgb_m.fit(X, y_log, verbose=False)
        xgb_models.append(xgb_m)

        rp      = best_params["ridge"] or {}
        ridge_m = Pipeline([("scaler", StandardScaler()), ("ridge", Ridge(alpha=rp.get("alpha", 10.0)))])
        ridge_m.fit(X, y_log)
        ridge_models.append(ridge_m)

    w = ensemble_weights

    def predict_raw(X_new):
        if not isinstance(X_new, pd.DataFrame):
            X_new = pd.DataFrame(X_new, columns=feature_cols)
        lgb_p   = np.mean([inv_transform(m.predict(X_new)) for m in lgb_models],   axis=0)
        lgb_q_p = np.mean([inv_transform(m.predict(X_new)) for m in lgb_q_models], axis=0)
        xgb_p   = np.mean([inv_transform(m.predict(X_new)) for m in xgb_models],   axis=0)
        ridge_p = np.mean([inv_transform(m.predict(X_new)) for m in ridge_models], axis=0)
        return (w[0]*lgb_p + w[1]*lgb_q_p + w[2]*xgb_p + w[3]*ridge_p).clip(0)

    return predict_raw, lgb_models, lgb_q_models, xgb_models, ridge_models

# =============================================================================
# STEP 10 — ISOTONIC CALIBRATION
# =============================================================================

def fit_isotonic_calibrator(oof_df, ensemble_weights):
    print("\n=== Fitting isotonic calibration ===")
    w = ensemble_weights
    oof_blend = (
        w[0]*oof_df["lgb"] + w[1]*oof_df["lgb_q"] +
        w[2]*oof_df["xgb"] + w[3]*oof_df["ridge"]
    ).values

    cal = IsotonicRegression(out_of_bounds="clip")
    cal.fit(oof_blend, oof_df["true"].values)

    raw_rmse = np.sqrt(mean_squared_error(oof_df["true"], oof_blend))
    cal_rmse = np.sqrt(mean_squared_error(oof_df["true"], cal.predict(oof_blend)))
    print(f"  OOF RMSE before calibration: {raw_rmse:.2f} L")
    print(f"  OOF RMSE after  calibration: {cal_rmse:.2f} L")
    return cal

# =============================================================================
# STEP 11 — POST-PROCESSING STACK
# =============================================================================

def build_scale_correction(train_df, feature_cols, predict_raw, calibrator):
    print("\n=== Computing vehicle × shift scale correction ===")
    clean = train_df[train_df["fuel_consumption_L"].notna()].copy()
    for col in feature_cols:
        if col not in clean.columns: clean[col] = 0

    clean["pred_cal"] = calibrator.predict(predict_raw(clean[feature_cols])).clip(0)

    vs_scale = (
        clean.groupby(["vehicle", "shift"])
        .apply(lambda g: g["fuel_consumption_L"].mean() / (g["pred_cal"].mean() + 1e-9))
        .clip(0.85, 1.20).rename("scale_vs").reset_index()
    )
    v_scale = (
        clean.groupby("vehicle")
        .apply(lambda g: g["fuel_consumption_L"].mean() / (g["pred_cal"].mean() + 1e-9))
        .clip(0.85, 1.20).rename("scale_v").reset_index()
    )
    overall = float(np.clip(
        clean["fuel_consumption_L"].mean() / (clean["pred_cal"].mean() + 1e-9), 0.85, 1.20
    ))

    vs_dict = {(r["vehicle"], r["shift"]): r["scale_vs"] for _, r in vs_scale.iterrows()}
    v_dict  = {r["vehicle"]: r["scale_v"] for _, r in v_scale.iterrows()}
    return vs_dict, v_dict, overall


def build_vehicle_clip_bounds(train_df):
    clean  = train_df[train_df["fuel_consumption_L"].notna()]
    bounds = clean.groupby("vehicle")["fuel_consumption_L"].agg(["min", "max"])
    return {v: {"lower": row["min"] * 0.5, "upper": row["max"] * 1.5}
            for v, row in bounds.iterrows()}


def apply_ewma_blend(preds_df, ewma_col="ewma_fuel_prior", blend=EWMA_BLEND):
    prior = preds_df[ewma_col].fillna(preds_df["fuel_consumed"])
    return (preds_df["fuel_consumed"] * (1 - blend) + prior * blend).clip(0)

# =============================================================================
# STEP 12 — PREDICT & SUBMIT
# =============================================================================

def predict_and_submit(predict_raw, calibrator, test_df_raw, targets_daily,
                       enc_maps, global_enc_mean, feature_cols,
                       vs_dict, v_dict, overall_scale, clip_bounds,
                       vehicle_mean, overall_median):
    print("\n=== Generating shift-level predictions ===")

    test_df = apply_full_encodings(test_df_raw, enc_maps, global_enc_mean)
    test_df = add_ewma_prior(test_df, targets_daily)

    for col in feature_cols:
        if col not in test_df.columns:
            test_df[col] = 0

    raw_preds = predict_raw(test_df[feature_cols])
    cal_preds = calibrator.predict(raw_preds).clip(0)
    test_df["fuel_consumed"] = cal_preds

    factors = test_df.apply(
        lambda r: vs_dict.get((r["vehicle"], r["shift"]),
                              v_dict.get(r["vehicle"], overall_scale)), axis=1
    )
    test_df["fuel_consumed"] = (test_df["fuel_consumed"] * factors).clip(0)
    test_df["fuel_consumed"] = apply_ewma_blend(test_df)

    # Hard zero physics override
    inactive_mask = (test_df["total_travel_hr"] < 0.1) & (test_df["total_idle_hr"] < 0.1)
    test_df.loc[inactive_mask, "fuel_consumed"] = 0.0

    def clip_row(r):
        b = clip_bounds.get(r["vehicle"], {"lower": 0.0, "upper": 1000.0})
        return float(np.clip(r["fuel_consumed"], b["lower"], b["upper"]))
    test_df["fuel_consumed"] = test_df.apply(clip_row, axis=1)

    zero_mask = test_df["fuel_consumed"] <= 0
    if zero_mask.any():
        test_df.loc[zero_mask, "fuel_consumed"] = (
            test_df.loc[zero_mask, "vehicle"].map(vehicle_mean).fillna(overall_median)
        )

    id_map = pd.read_csv(ID_MAPPING)
    id_map["date"] = pd.to_datetime(id_map["date"]).dt.date.astype(str)

    sub = id_map.merge(
        test_df[["vehicle", "op_day", "shift", "fuel_consumed"]],
        left_on=["vehicle", "date", "shift"],
        right_on=["vehicle", "op_day", "shift"],
        how="left",
    )
    still_missing = sub["fuel_consumed"].isna()
    if still_missing.any():
        sub.loc[still_missing, "fuel_consumed"] = (
            sub.loc[still_missing, "vehicle"].map(vehicle_mean).fillna(overall_median)
        )

    sub = sub[["id", "fuel_consumed"]].copy()
    sub["fuel_consumed"] = sub["fuel_consumed"].clip(0)
    sub_path = OUTPUT_DIR / "submission.csv"
    sub.to_csv(sub_path, index=False)
    print(f"\n  {len(sub):,} predictions → {sub_path}")
    return sub

# =============================================================================
# MAIN
# =============================================================================

if __name__ == "__main__":

    print("\n=== Loading train telemetry ===")
    df_train = load_telemetry(TRAIN_FILES)
    print("\n=== Loading test telemetry ===")
    df_test  = load_telemetry(TEST_FILES)

    print("\n=== Building train features ===")
    train_feats = build_features(df_train)
    print("\n=== Building test features ===")
    test_feats  = build_features(df_test)
    del df_train, df_test

    targets_raw   = pd.read_csv(TARGETS_FILE)
    targets_daily = targets_raw[["vehicle", "op_day", "fuel_consumption_L"]].copy()

    print("\n=== Building activity-weighted shift targets ===")
    targets_shift = build_shift_targets(targets_daily, train_feats)
    train_feats   = train_feats.merge(
        targets_shift, on=["vehicle", "op_day", "shift"], how="left"
    )

    print("\n=== Adding lag features ===")
    train_feats = add_lag_features(train_feats)
    test_feats  = add_lag_features(test_feats)

    vehicle_mean = (
        targets_raw[targets_raw["fuel_consumption_L"] > 0]
        .groupby("vehicle")["fuel_consumption_L"].mean()
        .div(N_SHIFTS).to_dict()
    )
    overall_median = targets_raw["fuel_consumption_L"].median() / N_SHIFTS

    feature_cols_base = get_feature_cols(train_feats)
    ensemble_rmse, fold_results, best_params, ensemble_weights, oof_df = walk_forward_cv(
        train_feats, targets_daily, feature_cols_base
    )
    oof_df.to_csv(OUTPUT_DIR / "oof_predictions.csv", index=False)

    print("\n=== Preparing full-train encodings ===")
    enc_maps, global_enc_mean = build_full_target_encodings(
        train_feats[train_feats["fuel_consumption_L"].notna()]
    )
    train_feats = apply_full_encodings(train_feats, enc_maps, global_enc_mean)
    train_feats = add_ewma_prior(train_feats, targets_daily)

    feature_cols = get_feature_cols(train_feats)
    for col in feature_cols:
        if col not in test_feats.columns:
            test_feats[col] = 0

    predict_raw_fn, lgb_m, lgb_q_m, xgb_m, ridge_m = train_final_ensemble(
        train_feats, feature_cols, best_params, ensemble_weights
    )
    calibrator = fit_isotonic_calibrator(oof_df, ensemble_weights)
    vs_dict, v_dict, overall_scale = build_scale_correction(
        train_feats, feature_cols, predict_raw_fn, calibrator
    )
    clip_bounds = build_vehicle_clip_bounds(train_feats)

    submission = predict_and_submit(
        predict_raw_fn, calibrator, test_feats, targets_daily,
        enc_maps, global_enc_mean, feature_cols,
        vs_dict, v_dict, overall_scale, clip_bounds,
        vehicle_mean, overall_median,
    )

    print("\n✓ Pipeline complete.")


=== Loading train telemetry ===
  Loading telemetry_2026-01-01_2026-01-10.parquet ... 4,334,181 rows
  Loading telemetry_2026-01-11_2026-01-20.parquet ... 4,208,802 rows
  Loading telemetry_2026-02-01_2026-02-10.parquet ... 4,711,861 rows
  Loading telemetry_2026-02-11_2026-02-20.parquet ... 5,031,978 rows
  Loading telemetry_2026-03-01_2026-03-11.parquet ... 4,991,055 rows
  GPS filter removed 265,683 rows (1.5%)
  → 17,794,920 rows | 30 dumpers | 131 operators

=== Loading test telemetry ===
  Loading telemetry_2026-01-21_2026-01-31.parquet ... 4,660,896 rows
  Loading telemetry_2026-02-21_2026-02-28.parquet ... 3,866,175 rows
  Loading telemetry_2026-03-12_2026-03-20.parquet ... 4,019,313 rows
  GPS filter removed 137,627 rows (1.4%)
  → 9,556,380 rows | 30 dumpers | 120 operators

=== Building train features ===
  Engineering shift-level features ...
  → 4,468 shift-rows | 28 columns

=== Building test features ===
  Engineering shift-level features ...
  → 2,431 shift-rows | 28 c

/tmp/ipykernel_55/665227616.py:418: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(vehicle_ewma).reset_index(drop=True)
/tmp/ipykernel_55/665227616.py:418: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(vehicle_ewma).reset_index(drop=True)


  Computing EWMA rolling fuel prior ...

  ── Fold 1  (train=789, val=405, features=25) ──
    Tuning LGB RMSE ...
    LGB RMSE  = 62.27 L
    Tuning LGB Quantile ...


[LightGBM] [Fatal] Reducing `min_data_in_leaf` with `feature_pre_filter=true` may cause unexpected behaviour for features that were pre-filtered by the larger `min_data_in_leaf`.
You need to set `feature_pre_filter=false` to dynamically change the `min_data_in_leaf`.
[LightGBM] [Fatal] Reducing `min_data_in_leaf` with `feature_pre_filter=true` may cause unexpected behaviour for features that were pre-filtered by the larger `min_data_in_leaf`.
You need to set `feature_pre_filter=false` to dynamically change the `min_data_in_leaf`.


    LGB Q50   = 62.20 L
    Tuning XGBoost ...
    XGB RMSE  = 62.10 L
    Tuning Ridge ...
    Ridge RMSE = 95.18 L
  Computing EWMA rolling fuel prior ...
  Computing EWMA rolling fuel prior ...


...

  Run 2/5  (seed=49)
  Run 3/5  (seed=56)
  Run 4/5  (seed=63)
  Run 5/5  (seed=70)

=== Fitting isotonic calibration ===
  OOF RMSE before calibration: 76.92 L
  OOF RMSE after  calibration: 74.89 L

=== Computing vehicle × shift scale correction ===


/tmp/ipykernel_55/665227616.py:835: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g["fuel_consumption_L"].mean() / (g["pred_cal"].mean() + 1e-9))
/tmp/ipykernel_55/665227616.py:840: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g["fuel_consumption_L"].mean() / (g["pred_cal"].mean() + 1e-9))
/tmp/ipykernel_55/665227616.py:418: FutureWarning: DataFrameGroupBy.apply operated on the grou


=== Generating shift-level predictions ===
  Computing EWMA rolling fuel prior ...


/tmp/ipykernel_55/665227616.py:526: RuntimeWarning: overflow encountered in expm1
  def inv_transform(y): return np.expm1(y).clip(0)
/tmp/ipykernel_55/665227616.py:526: RuntimeWarning: overflow encountered in expm1
  def inv_transform(y): return np.expm1(y).clip(0)
/tmp/ipykernel_55/665227616.py:526: RuntimeWarning: overflow encountered in expm1
  def inv_transform(y): return np.expm1(y).clip(0)
/tmp/ipykernel_55/665227616.py:526: RuntimeWarning: overflow encountered in expm1
  def inv_transform(y): return np.expm1(y).clip(0)
/tmp/ipykernel_55/665227616.py:526: RuntimeWarning: overflow encountered in expm1
  def inv_transform(y): return np.expm1(y).clip(0)


ValueError: Input contains infinity or a value too large for dtype('float64').

In [1]:
import pandas as pd

# Replace with your actual file path
file_path = '/kaggle/input/datasets/nandishrajukori/train-targets-csv/train_targets.csv'

# Read the CSV
df = pd.read_csv(file_path)

# Save a copy to the /kaggle/working directory
df.to_csv('/kaggle/working/my_copy.csv', index=False)
